In [ ]:
"""
create_grid.py

Purpose:
    Generate a 1° x 1° spatial grid for the Philippines,
    mask/clip the grid using a Philippine land boundary,
    calculate land coverage, assign representative coordinates,
    and export the final spatial grid and metadata.

Outputs:
    data/processed/philippines_grid.geojson
    data/processed/grid_metadata.csv
"""
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import box 


# ============================================================
# USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# Philippine boundary file
# ------------------------------------------------------------
BOUNDARY_FILE = Path(
    "data/boundaries/Philippines.shp"
)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "data/processed"
)

# ------------------------------------------------------------
# Geographic extent used to GENERATE candidate cells
#
# IMPORTANT:
# This is a broad bounding box, NOT the Philippine
# political boundary itself.
# ------------------------------------------------------------

MIN_LON = 116
MAX_LON = 128

MIN_LAT = 4
MAX_LAT = 22

# ------------------------------------------------------------
# Grid resolution
# ------------------------------------------------------------

GRID_SIZE_DEGREES = 1.0

# ------------------------------------------------------------
# Land coverage threshold
#
# 0.0 = retain every cell that contains any Philippine land
#
# Example:
# 10.0 = retain only cells with at least 10% land
#
# Recommended initially:
# 0.0
# ------------------------------------------------------------

MIN_LAND_COVERAGE = 0.0

# ------------------------------------------------------------
# Coordinate reference systems
# ------------------------------------------------------------

WGS84 = "EPSG:4326"

# Equal-area projection for area calculations
EQUAL_AREA_CRS = "EPSG:6933"


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CHECK INPUT FILE
# ============================================================

if not BOUNDARY_FILE.exists():

    raise FileNotFoundError(
        f"\nPhilippine boundary file not found:\n"
        f"{BOUNDARY_FILE}\n\n"
        f"Please place your boundary file in:\n"
        f"data/boundaries/\n"
    )


# ============================================================
# LOAD PHILIPPINE LAND BOUNDARY
# ============================================================

print("\nLoading Philippine boundary...")

philippines = gpd.read_file(
    BOUNDARY_FILE
)

print(
    f"Boundary CRS: {philippines.crs}"
)

print(
    f"Boundary features: {len(philippines)}"
)


# ============================================================
# CHECK CRS
# ============================================================

if philippines.crs is None:

    raise ValueError(
        "\nThe Philippine boundary does not have a CRS.\n"
        "Please define its CRS before running this script."
    )


# Convert to WGS84 longitude/latitude
philippines = philippines.to_crs(
    WGS84
)


# ============================================================
# CLEAN INVALID GEOMETRIES
# ============================================================

print("\nChecking geometries...")

invalid_count = (
    ~philippines.geometry.is_valid
).sum()

print(
    f"Invalid geometries found: {invalid_count}"
)

if invalid_count > 0:

    print(
        "Attempting to repair invalid geometries..."
    )

    philippines["geometry"] = (
        philippines.geometry.make_valid()
    )


# ============================================================
# DISSOLVE PHILIPPINE LAND INTO ONE GEOMETRY
# ============================================================

print("\nDissolving Philippine land boundary...")

philippines_land = (
    philippines.geometry.union_all()
)

print("Philippine land boundary prepared.")


# ============================================================
# GENERATE CANDIDATE 1° × 1° GRID
# ============================================================

print("\nGenerating candidate grid...")

cells = []

lon = MIN_LON

while lon < MAX_LON:

    lat = MIN_LAT

    while lat < MAX_LAT:

        cell = box(
            lon,
            lat,
            lon + GRID_SIZE_DEGREES,
            lat + GRID_SIZE_DEGREES
        )

        grid_id = (
            f"G_LAT-{int(lat)}_LON-{int(lon)}"
        )

        cells.append({

            "grid_id": grid_id,

            "lat_min": lat,
            "lat_max": lat + GRID_SIZE_DEGREES,

            "lon_min": lon,
            "lon_max": lon + GRID_SIZE_DEGREES,

            "geometry": cell
        })

        lat += GRID_SIZE_DEGREES

    lon += GRID_SIZE_DEGREES


candidate_grid = gpd.GeoDataFrame(
    cells,
    crs=WGS84
)

print(
    f"Candidate cells generated: "
    f"{len(candidate_grid)}"
)


# ============================================================
# FIND CELLS THAT INTERSECT PHILIPPINE LAND
# ============================================================

print(
    "\nIdentifying cells that intersect "
    "Philippine land..."
)

candidate_grid["land_intersection"] = (
    candidate_grid.geometry.intersection(
        philippines_land
    )
)


# Remove cells with no Philippine land
grid = candidate_grid[
    ~candidate_grid["land_intersection"].is_empty
].copy()

grid = grid[
    grid["land_intersection"].notna()
].copy()

grid.reset_index(
    drop=True,
    inplace=True
)

print(f"Final retained cells: {len(grid)}")

# ------------------------------------------------------------
# Assign sequential grid numbers
# ------------------------------------------------------------

grid = grid.sort_values(
    by=["lat_min", "lon_min"]
).reset_index(drop=True)

grid["grid_number"] = range(1, len(grid) + 1)

print("\nGrid numbers assigned:")
print(
    grid[
        ["grid_number", "grid_id", "lat_min", "lon_min"]
    ].head(10).to_string(index=False)
)

print(
    f"Cells intersecting Philippine land: "
    f"{len(grid)}"
)


# ============================================================
# CALCULATE AREAS
# ============================================================

print(
    "\nCalculating cell and land areas..."
)

# ------------------------------------------------------------
# Original grid in equal-area projection
# ------------------------------------------------------------

grid_equal_area = grid[
    [
        "grid_id",
        "geometry"
    ]
].copy()

grid_equal_area = grid_equal_area.to_crs(
    EQUAL_AREA_CRS
)


# ------------------------------------------------------------
# Philippine land intersections
# ------------------------------------------------------------

land_intersections = gpd.GeoSeries(
    grid["land_intersection"],
    crs=WGS84
)

land_intersections_equal_area = (
    land_intersections.to_crs(
        EQUAL_AREA_CRS
    )
)


# ------------------------------------------------------------
# Areas in square kilometers
# ------------------------------------------------------------

grid["cell_area_km2"] = (
    grid_equal_area.geometry.area
    / 1_000_000
)

grid["land_area_km2"] = (
    land_intersections_equal_area.area
    / 1_000_000
)


# ------------------------------------------------------------
# Land coverage percentage
# ------------------------------------------------------------

grid["land_coverage_pct"] = (

    grid["land_area_km2"]
    /
    grid["cell_area_km2"]
    *
    100

)


# ============================================================
# APPLY LAND COVERAGE THRESHOLD
# ============================================================

print(
    f"\nApplying land coverage threshold: "
    f"{MIN_LAND_COVERAGE}%"
)

grid = grid[
    grid["land_coverage_pct"]
    >= MIN_LAND_COVERAGE
].copy()

grid.reset_index(
    drop=True,
    inplace=True
)

print(
    f"Final retained cells: {len(grid)}"
)


# ============================================================
# CREATE REPRESENTATIVE POINT
# ============================================================

print(
    "\nGenerating representative coordinates..."
)

# A representative point is guaranteed to lie inside
# the geometry of the Philippine land intersection.

grid["representative_point"] = (
    grid["land_intersection"]
    .representative_point()
)


# ============================================================
# EXTRACT LATITUDE AND LONGITUDE
# ============================================================

grid["query_longitude"] = (
    grid["representative_point"].x
)

grid["query_latitude"] = (
    grid["representative_point"].y
)


# ============================================================
# CALCULATE ORIGINAL CELL CENTER
# ============================================================

grid["cell_center_longitude"] = (
    (
        grid["lon_min"]
        +
        grid["lon_max"]
    )
    / 2
)

grid["cell_center_latitude"] = (
    (
        grid["lat_min"]
        +
        grid["lat_max"]
    )
    / 2
)


# ============================================================
# CREATE CELL LABEL
# ============================================================

grid["cell_label"] = (
    "Lat "
    + grid["lat_min"].astype(str)
    + "–"
    + grid["lat_max"].astype(str)
    + ", Lon "
    + grid["lon_min"].astype(str)
    + "–"
    + grid["lon_max"].astype(str)
)


# ============================================================
# ROUND NUMERICAL VALUES
# ============================================================

grid["cell_area_km2"] = (
    grid["cell_area_km2"].round(3)
)

grid["land_area_km2"] = (
    grid["land_area_km2"].round(3)
)

grid["land_coverage_pct"] = (
    grid["land_coverage_pct"].round(3)
)

grid["query_latitude"] = (
    grid["query_latitude"].round(6)
)

grid["query_longitude"] = (
    grid["query_longitude"].round(6)
)

grid["cell_center_latitude"] = (
    grid["cell_center_latitude"].round(6)
)

grid["cell_center_longitude"] = (
    grid["cell_center_longitude"].round(6)
)


# ============================================================
# PREPARE FINAL GEOJSON
# ============================================================

print(
    "\nPreparing GeoJSON..."
)

# The GeoJSON should contain the ORIGINAL 1° × 1° cells,
# not the clipped land fragments.
#
# This preserves your regular spatial analysis grid.

final_grid = grid.drop(
    columns=[
        "land_intersection",
        "representative_point"
    ]
).copy()

final_grid = final_grid.to_crs(
    WGS84
)


# ============================================================
# SAVE GEOJSON
# ============================================================

geojson_file = (
    OUTPUT_DIR
    /
    "philippines_grid.geojson"
)

final_grid.to_file(
    geojson_file,
    driver="GeoJSON"
)

print(
    f"\nGeoJSON saved to:\n"
    f"{geojson_file}"
)


# ============================================================
# CREATE METADATA TABLE
# ============================================================

metadata_columns = [

    "grid_number",

    "grid_id",

    "cell_label",

    "lat_min",
    "lat_max",

    "lon_min",
    "lon_max",

    "cell_center_latitude",
    "cell_center_longitude",

    "query_latitude",
    "query_longitude",

    "cell_area_km2",

    "land_area_km2",

    "land_coverage_pct"
]


metadata = final_grid[
    metadata_columns
].copy()


# ============================================================
# SORT GRID IDS
# ============================================================

metadata = metadata.sort_values(
    by=[
        "lat_min",
        "lon_min"
    ]
).reset_index(
    drop=True
)


# ============================================================
# SAVE CSV
# ============================================================

csv_file = (
    OUTPUT_DIR
    /
    "grid_metadata.csv"
)

metadata.to_csv(
    csv_file,
    index=False
)

print(
    f"Metadata CSV saved to:\n"
    f"{csv_file}"
)


# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("GRID GENERATION COMPLETE")
print("=" * 60)

print(
    f"Candidate cells:       "
    f"{len(candidate_grid)}"
)

print(
    f"Land-intersecting:     "
    f"{len(grid)}"
)

print(
    f"Final retained cells:  "
    f"{len(final_grid)}"
)

print(
    f"Grid resolution:       "
    f"{GRID_SIZE_DEGREES}° × "
    f"{GRID_SIZE_DEGREES}°"
)

print(
    f"Minimum land coverage: "
    f"{MIN_LAND_COVERAGE}%"
)

print(
    f"CRS:                   "
    f"{WGS84}"
)

print(
    f"Area CRS:              "
    f"{EQUAL_AREA_CRS}"
)

print("\nOutputs:")

print(
    f"  {geojson_file}"
)

print(
    f"  {csv_file}"
)

print("\nFirst 10 grid cells:")

print(
    metadata.head(10).to_string(
        index=False
    )
)

print("\nDone.")


Loading Philippine boundary...
Boundary CRS: EPSG:32651
Boundary features: 3640

Checking geometries...
Invalid geometries found: 0

Dissolving Philippine land boundary...
Philippine land boundary prepared.

Generating candidate grid...
Candidate cells generated: 216

Identifying cells that intersect Philippine land...
Final retained cells: 93

Grid numbers assigned:
 grid_number         grid_id  lat_min  lon_min
           1 G_LAT-4_LON-119      4.0    119.0
           2 G_LAT-4_LON-120      4.0    120.0
           3 G_LAT-5_LON-119      5.0    119.0
           4 G_LAT-5_LON-120      5.0    120.0
           5 G_LAT-5_LON-121      5.0    121.0
           6 G_LAT-5_LON-124      5.0    124.0
           7 G_LAT-5_LON-125      5.0    125.0
           8 G_LAT-6_LON-118      6.0    118.0
           9 G_LAT-6_LON-120      6.0    120.0
          10 G_LAT-6_LON-121      6.0    121.0
Cells intersecting Philippine land: 93

Calculating cell and land areas...

Applying land coverage threshold: 0.